# Steerable Chatbots — vector construction

Builds the style vector for **"Steerable Chatbots: Personalizing LLMs with Preference-Based Activation Steering"** ([arXiv:2505.04260](https://arxiv.org/abs/2505.04260v2)) on Qwen2.5-1.5B-Instruct.

Captures hidden states over adult-oriented vs child-oriented activity recommendations, then fits a linear-probe control vector (`extract_linear_probe_control_vector`) exported as `style-probe.gguf`. Consumed by `steer.ipynb`.

In [ ]:
# positive adult
adults_oriented_examples = [
    "Visit the rooftop bar downtown for stunning views and a curated selection of craft cocktails.",
    "Plan a day of wine tasting at the region's top vineyards, complete with private tours.",
    "Book a spa day with massages, saunas, and relaxation areas exclusively for adults.",
    "Attend a live jazz performance at the speakeasy-style venue known for its intimate ambiance.",
    "Join a mixology workshop to learn the secrets behind making perfect cocktails at home.",
    "Take a sunset cruise offering drinks and live music in a serene, adults-only setting.",
    "Visit the cigar lounge downtown, featuring an extensive selection and a cozy atmosphere.",
    "Book a brewery tour with tastings of seasonal beers and behind-the-scenes insights from brewmasters.",
    "Attend an adults-only trivia night at the local pub it's a fun way to test your knowledge.",
    "Visit the distillery for a tour and a guided whiskey tasting experience."
]

# negative child
kids_oriented_examples = [
    "Visit the interactive science museum with hands-on exhibits perfect for kids of all ages.",
    "Plan a day at the local zoo, featuring kid-friendly animal encounters and feeding sessions.",
    "Explore the city's largest playground with climbing structures, slides, and picnic areas for families.",
    "Spend time at the local aquarium, known for its touch tanks and playful sea otters.",
    "Visit the city's amusement park with rides designed specifically for younger kids and toddlers.",
    "Head to the local library for storytime sessions and engaging activities for kids.",
    "Visit the butterfly garden it's colorful, educational, and a hit with younger children.",
    "Take a scenic train ride that offers kid-friendly entertainment and stunning views along the way.",
    "Visit the local zookeeper for a behind-the-scenes tour kids love learning about animals up close.",
    "Spend time at the nearby petting farm, where kids can feed and interact with friendly animals.",
    "Explore the city's famous ice cream parlor, offering fun flavors and mini cones for little ones."
]

In [ ]:
import os

import easysteer.hidden_states as hs
from vllm import LLM

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

MODEL = "/home/shenyl/hf/model/Qwen/Qwen2.5-1.5B-Instruct/"  # Qwen/Qwen2.5-1.5B-Instruct

# Capture needs eager execution and prefix caching off: cache-hit
# tokens are never recomputed, so their hidden states can't be captured.
llm = LLM(
    model=MODEL,
    tensor_parallel_size=1,
    enforce_eager=True,
    enable_prefix_caching=False,
)

all_hidden_states, outputs = hs.get_all_hidden_states_generate(
    llm, adults_oriented_examples + kids_oriented_examples
)

In [ ]:
from easysteer.steer import extract_linear_probe_control_vector, StatisticalControlVector
control_vector = extract_linear_probe_control_vector(
    all_hidden_states=all_hidden_states, 
    positive_indices=list(range(10)), 
    negative_indices=list(range(10,20)), 
    model_type="qwen2.5",
    token_pos=-1, 
    normalize=True
)
control_vector.export_gguf("style-probe.gguf")